# Análisis de Estrategias para Ganar en Monopoly

## Objetivo del Proyecto
El objetivo de este proyecto es investigar si existen estrategias óptimas para ganar en el juego de Monopoly mediante simulaciones computacionales. Utilizamos un enfoque basado en datos para simular partidas de Monopoly, analizar los resultados y responder preguntas clave sobre las mejores estrategias y propiedades para maximizar las probabilidades de victoria.

### Pregunta Principal
**¿Existen estrategias para ganar en Monopoly?**

### Subpreguntas
1. **¿Cuáles son las propiedades que te dan más probabilidades de ganar?**
   - Identificamos las propiedades más visitadas y las más rentables mediante un mapa de calor de frecuencias.
2. **¿Qué estrategias de compra y negociación son más efectivas?**
   - Analizamos diferentes comportamientos de los jugadores (por ejemplo, comprar agresivamente o ser más conservador).
3. **¿Cómo influyen las casillas especiales (impuestos, cárceles, etc.) en las probabilidades de ganar?**
   - Evaluamos el impacto de las casillas no comprables en el flujo del juego.
4. **¿Cuánto tiempo promedio toma una partida de Monopoly bajo diferentes estrategias?**
   - Calculamos la duración promedio de las partidas para entender cómo las estrategias afectan la dinámica del juego.

## Descripción del Código
Este proyecto incluye un simulador de Monopoly que permite jugar partidas con diferentes estrategias de jugadores. Generamos visualizaciones para analizar los resultados, incluyendo:

- **Mapa de Calor del Tablero**: Muestra las casillas más visitadas (en una imagen estática generada con Matplotlib).
- Gráficos de porcentaje de victorias, evolución del dinero, propiedades más comunes.

El mapa de calor estático muestra las frecuencias de visita a cada casilla, con etiquetas rotadas, nombres divididos en dos líneas para mayor legibilidad, y un diseño estilizado que incluye barras de color para las propiedades y un fondo beige.



Genere una clase para casilla que tenga la informacion para casilla  

In [86]:
from IPython import get_ipython
from IPython.display import display

In [87]:
class Casilla:
    def __init__(self, nombre, tipo, costo=0, renta=0, costo_casa=0, renta_con_casas=None, costo_hotel=0, renta_hotel=0, grupo=None):
        self.nombre = nombre
        self.tipo = tipo
        self.costo = costo
        self.renta = renta
        self.propietario = None
        self.casas = 0
        self.hoteles = 0
        self.costo_casa = costo_casa
        self.costo_hotel = costo_hotel or costo_casa  # Default to costo_casa if not specified
        self.renta_con_casas = renta_con_casas or [0] * 4  # Explicit rents for 1-4 houses
        self.renta_hotel = renta_hotel
        self.grupo = grupo
        self.hipotecada = False
        self.valor_hipoteca = costo // 2 if costo > 0 else 0

    def calcular_renta(self, tablero):
        if self.hipotecada:
            return 0
        if self.hoteles > 0:
            return self.renta_hotel
        elif self.casas > 0:
            return self.renta_con_casas[self.casas - 1]
        else:
            # Double rent for monopoly
            if self.grupo and self.propietario:
                grupo = [p for p in tablero.casillas if p.grupo == self.grupo and p.tipo == "propiedad"]
                if all(p.propietario == self.propietario for p in grupo):
                    return self.renta * 2
            return self.renta

In [88]:
class Tablero:
    def __init__(self):
        print("Creando tablero...")
        self.casillas = self.crear_tablero()
        self.grupos = self.crear_grupos()
        self.posicion_carcel = next(i for i, c in enumerate(self.casillas) if c.nombre == "Cárcel / Solo de visita")

    def crear_grupos(self):
        grupos = {}
        for casilla in self.casillas:
            if casilla.grupo:
                if casilla.grupo not in grupos:
                    grupos[casilla.grupo] = []
                grupos[casilla.grupo].append(casilla)
        return grupos

    def crear_tablero(self):
        print("Inicializando las casillas...")
        return [
            Casilla("Salida", "especial"),
            Casilla("Mediterranean Avenue", "propiedad", costo=60, renta=2, costo_casa=50, costo_hotel=50, renta_con_casas=[10, 30, 90, 160], renta_hotel=250, grupo="morado"),
            Casilla("Comunidad", "comunidad"),
            Casilla("Baltic Avenue", "propiedad", costo=60, renta=4, costo_casa=50, costo_hotel=50, renta_con_casas=[20, 60, 180, 320], renta_hotel=450, grupo="morado"),
            Casilla("Impuesto sobre la renta", "impuesto", costo=200),
            Casilla("Ferrocarril Reading", "ferrocarril", costo=200, renta=25),
            Casilla("Oriental Avenue", "propiedad", costo=100, renta=6, costo_casa=50, costo_hotel=50, renta_con_casas=[30, 90, 270, 400], renta_hotel=550, grupo="celeste"),
            Casilla("Suerte", "suerte"),
            Casilla("Vermont Avenue", "propiedad", costo=100, renta=6, costo_casa=50, costo_hotel=50, renta_con_casas=[30, 90, 270, 400], renta_hotel=550, grupo="celeste"),
            Casilla("Connecticut Avenue", "propiedad", costo=120, renta=8, costo_casa=50, costo_hotel=50, renta_con_casas=[40, 100, 300, 450], renta_hotel=600, grupo="celeste"),
            Casilla("Cárcel / Solo de visita", "especial"),
            Casilla("St. Charles Place", "propiedad", costo=140, renta=10, costo_casa=100, costo_hotel=100, renta_con_casas=[50, 150, 450, 625], renta_hotel=750, grupo="rosado"),
            Casilla("Comunidad", "comunidad"),
            Casilla("States Avenue", "propiedad", costo=140, renta=10, costo_casa=100, costo_hotel=100, renta_con_casas=[50, 150, 450, 625], renta_hotel=750, grupo="rosado"),
            Casilla("Virginia Avenue", "propiedad", costo=160, renta=12, costo_casa=100, costo_hotel=100, renta_con_casas=[60, 180, 500, 700], renta_hotel=900, grupo="rosado"),
            Casilla("Ferrocarril Pennsylvania", "ferrocarril", costo=200, renta=25),
            Casilla("St. James Place", "propiedad", costo=180, renta=14, costo_casa=100, costo_hotel=100, renta_con_casas=[70, 200, 550, 750], renta_hotel=950, grupo="naranja"),
            Casilla("Suerte", "suerte"),
            Casilla("Tennessee Avenue", "propiedad", costo=180, renta=14, costo_casa=100, costo_hotel=100, renta_con_casas=[70, 200, 550, 750], renta_hotel=950, grupo="naranja"),
            Casilla("New York Avenue", "propiedad", costo=200, renta=16, costo_casa=100, costo_hotel=100, renta_con_casas=[80, 220, 600, 800], renta_hotel=1000, grupo="naranja"),
            Casilla("Parada libre", "especial"),
            Casilla("Kentucky Avenue", "propiedad", costo=220, renta=18, costo_casa=150, costo_hotel=150, renta_con_casas=[90, 250, 700, 875], renta_hotel=1050, grupo="rojo"),
            Casilla("Suerte", "suerte"),
            Casilla("Indiana Avenue", "propiedad", costo=220, renta=18, costo_casa=150, costo_hotel=150, renta_con_casas=[90, 250, 700, 875], renta_hotel=1050, grupo="rojo"),
            Casilla("Illinois Avenue", "propiedad", costo=240, renta=20, costo_casa=150, costo_hotel=150, renta_con_casas=[100, 300, 750, 925], renta_hotel=1100, grupo="rojo"),
            Casilla("Ferrocarril B&O", "ferrocarril", costo=200, renta=25),
            Casilla("Atlantic Avenue", "propiedad", costo=260, renta=22, costo_casa=150, costo_hotel=150, renta_con_casas=[110, 330, 800, 975], renta_hotel=1150, grupo="amarillo"),
            Casilla("Ventnor Avenue", "propiedad", costo=260, renta=22, costo_casa=150, costo_hotel=150, renta_con_casas=[110, 330, 800, 975], renta_hotel=1150, grupo="amarillo"),
            Casilla("Compañía de agua", "servicio", costo=150, renta=0),
            Casilla("Marvin Gardens", "propiedad", costo=280, renta=24, costo_casa=150, costo_hotel=150, renta_con_casas=[120, 360, 850, 1025], renta_hotel=1200, grupo="amarillo"),
            Casilla("Ve a la cárcel", "especial"),
            Casilla("Pacific Avenue", "propiedad", costo=300, renta=26, costo_casa=200, costo_hotel=200, renta_con_casas=[130, 390, 900, 1100], renta_hotel=1275, grupo="verde"),
            Casilla("Carolina del Norte Avenue", "propiedad", costo=300, renta=26, costo_casa=200, costo_hotel=200, renta_con_casas=[130, 390, 900, 1100], renta_hotel=1275, grupo="verde"),
            Casilla("Comunidad", "comunidad"),
            Casilla("Avenida Pennsylvania", "propiedad", costo=320, renta=28, costo_casa=200, costo_hotel=200, renta_con_casas=[150, 450, 1000, 1200], renta_hotel=1400, grupo="verde"),
            Casilla("Ferrocarril Short Line", "ferrocarril", costo=200, renta=25),
            Casilla("Suerte", "suerte"),
            Casilla("Park Place", "propiedad", costo=350, renta=35, costo_casa=200, costo_hotel=200, renta_con_casas=[175, 500, 1100, 1300], renta_hotel=1500, grupo="azul oscuro"),
            Casilla("Impuesto de lujo", "impuesto", costo=100),
            Casilla("Boardwalk", "propiedad", costo=400, renta=50, costo_casa=200, costo_hotel=200, renta_con_casas=[200, 600, 1400, 1700], renta_hotel=2000, grupo="azul oscuro")
        ]

In [89]:
import random

class Jugador:
    def __init__(self, nombre, silencioso=True):
        self.nombre = nombre
        self.dinero = 1500
        self.propiedades = []
        self.posicion = 0
        self.en_carcel = False
        self.turnos_en_carcel = 0
        self.tiene_carta_salir_carcel = False  # Para cartas de "Salir de la cárcel"
        self.silencioso = silencioso

    def lanzar_dados(self):
        """Lanza dos dados y devuelve el resultado como una tupla."""
        return random.randint(1, 6), random.randint(1, 6)

    def mover(self, total_dados, tablero):
        """Mueve al jugador en el tablero y otorga $200 si pasa por la casilla de salida."""
        posicion_anterior = self.posicion  # Guarda la posición antes de moverse
        self.posicion = (self.posicion + total_dados) % len(tablero.casillas)
    
    # Si el jugador cayó exactamente en la casilla de salida
        if tablero.casillas[self.posicion].nombre == "Salida":
                self.dinero += 200
        if not self.silencioso:
            print(f"{self.nombre} cayó en la casilla 'Salida' y recibió $200. Nuevo saldo: {self.dinero}")
    
        return tablero.casillas[self.posicion]


    def tiene_todas_las_propiedades_del_grupo(self, grupo):
        return all(p.propietario == self for p in tablero.grupos[grupo])

    def puede_construir_casa(self, propiedad):
        """Verifica si puede construir una casa."""
        return (propiedad.propietario == self and
                self.tiene_todas_las_propiedades_del_grupo(propiedad.grupo) and
                propiedad.casas < 4 and
                self.dinero >= propiedad.costo_casa)

    def construir_casa(self, propiedad):
        """Construye una casa en la propiedad si es posible."""
        if self.puede_construir_casa(propiedad):
            self.dinero -= propiedad.costo_casa
            propiedad.casas += 1
            if not self.silencioso:
                print(f"{self.nombre} construyó una casa en {propiedad.nombre}.")
        else:
            if not self.silencioso:
                print(f"{self.nombre} no puede construir una casa en {propiedad.nombre}.")

    def puede_construir_hotel(self, propiedad):
        """Verifica si puede construir un hotel."""
        return propiedad.casas == 4 and propiedad.hoteles == 0 and self.dinero >= propiedad.costo_hotel

    def construir_hotel(self, propiedad):
        """Construye un hotel en la propiedad si es posible."""
        if self.puede_construir_hotel(propiedad):
            self.dinero -= propiedad.costo_hotel
            propiedad.hoteles += 1
            propiedad.casas = 0  # Se convierten en un hotel
            if not self.silencioso:
                print(f"{self.nombre} construyó un hotel en {propiedad.nombre}.")
        else:
            if not self.silencioso:
                print(f"{self.nombre} no puede construir un hotel en {propiedad.nombre}.")

    def gestionar_carcel(self):
        """Maneja la lógica cuando un jugador está en la cárcel."""
        if self.en_carcel:
            if self.tiene_carta_salir_carcel:
                self.en_carcel = False
                self.tiene_carta_salir_carcel = False
                if not self.silencioso:
                    print(f"{self.nombre} usó una carta de 'Salir de la cárcel'.")
            elif self.dinero >= 50:  # Pago de fianza
                self.dinero -= 50
                self.en_carcel = False
                if not self.silencioso:
                    print(f"{self.nombre} pagó $50 para salir de la cárcel.")
            else:
                self.turnos_en_carcel += 1
                if self.turnos_en_carcel == 3:
                    self.en_carcel = False
                    self.turnos_en_carcel = 0
                    if not self.silencioso:
                        print(f"{self.nombre} sale de la cárcel después de 3 turnos.")

    def ejecutar_turno(self, juego):
        """Ejecuta un turno, incluyendo los dobles y la cárcel."""
        if self.en_carcel:
            self.gestionar_carcel()
            return  # Si sigue en la cárcel, no juega

        turnos_consecutivos = 0
        while True:
            dado1, dado2 = self.lanzar_dados()
            total_dados = dado1 + dado2
            if dado1 == dado2:
                turnos_consecutivos += 1
                if turnos_consecutivos == 3:
                    self.en_carcel = True
                    self.posicion = juego.tablero.obtener_posicion_carcel()
                    if not self.silencioso:
                        print(f"{self.nombre} sacó tres dobles seguidos y va a la cárcel.")
                    break

            casilla = self.mover(total_dados, juego.tablero)

            # Manejo de propiedad
            if casilla.tipo == "propiedad" and casilla.propietario is None:
                juego.comprar_propiedad(self, casilla)
            elif casilla.tipo == "propiedad" and casilla.propietario != self:
                juego.pagar_renta(self, casilla)

            # Si no sacó dobles, termina su turno
            if dado1 != dado2:
                break

    def hipotecar_propiedad(self, propiedad):
        """Hipoteca una propiedad para obtener dinero."""
        if (propiedad.propietario == self and 
            not propiedad.hipotecada and 
            propiedad.casas == 0 and propiedad.hoteles == 0):  # No se puede hipotecar con construcciones
            propiedad.hipotecada = True
            self.dinero += propiedad.valor_hipoteca
            if not self.silencioso:
                print(f"{self.nombre} hipotecó {propiedad.nombre} por ${propiedad.valor_hipoteca}.")
            return True
        return False

    def deshipotecar_propiedad(self, propiedad):
        """Deshipoteca una propiedad pagando el valor + 10% de interés."""
        costo_deshipoteca = int(propiedad.valor_hipoteca * 1.1)  # +10% de interés
        if propiedad.propietario == self and propiedad.hipotecada and self.dinero >= costo_deshipoteca:
            propiedad.hipotecada = False
            self.dinero -= costo_deshipoteca
            if not self.silencioso:
                print(f"{self.nombre} deshipotecó {propiedad.nombre} por ${costo_deshipoteca}.")
            return True
        return False

    def pagar_deuda(self, monto, acreedor=None):
        """Intenta pagar una deuda, primero con dinero, luego hipotecando propiedades, y finalmente ofreciendo propiedades."""
        # Primero intenta pagar con dinero disponible
        if self.dinero >= monto:
            self.dinero -= monto
            if acreedor:
                acreedor.dinero += monto
            return True
        
        # Si no tiene suficiente dinero, intenta hipotecar propiedades
        while self.dinero < monto and any(p for p in self.propiedades if not p.hipotecada and p.casas == 0 and p.hoteles == 0):
            propiedad_a_hipotecar = min(
                [p for p in self.propiedades if not p.hipotecada and p.casas == 0 and p.hoteles == 0], 
                key=lambda x: x.renta, 
                default=None
            )
            if propiedad_a_hipotecar:
                self.hipotecar_propiedad(propiedad_a_hipotecar)
            else:
                break
        
        # Si aún no tiene suficiente, ofrece propiedades al acreedor
        if self.dinero < monto and acreedor and self.propiedades:
            # Ordenar propiedades por valor (de mayor a menor)
            propiedades_ordenadas = sorted(
                self.propiedades,
                key=lambda p: p.costo,
                reverse=True
            )
            
            for propiedad in propiedades_ordenadas:
                if propiedad.costo >= monto - self.dinero:
                    # Transferir la propiedad
                    self.transferir_propiedad(propiedad, acreedor)
                    # Ajustar el dinero (el acreedor "paga" la diferencia)
                    diferencia = propiedad.costo - (monto - self.dinero)
                    self.dinero = diferencia
                    acreedor.dinero -= diferencia
                    return True
            
            # Si ninguna propiedad cubre la deuda sola, ofrecer todas
            valor_total = sum(p.costo for p in self.propiedades)
            if valor_total + self.dinero >= monto:
                for propiedad in self.propiedades[:]:
                    self.transferir_propiedad(propiedad, acreedor)
                self.dinero += valor_total - monto
                acreedor.dinero -= valor_total - monto
                return True
        
        # Si no puede pagar de ninguna manera
        if self.dinero < monto:
            if not self.silencioso:
                print(f"{self.nombre} no pudo pagar ${monto} y está en bancarrota.")
            return False
        else:
            self.dinero -= monto
            if acreedor:
                acreedor.dinero += monto
            return True

    def transferir_propiedad(self, propiedad, nuevo_propietario):
        """Transfiere una propiedad a otro jugador."""
        if propiedad in self.propiedades:
            self.propiedades.remove(propiedad)
            nuevo_propietario.propiedades.append(propiedad)
            propiedad.propietario = nuevo_propietario
            # Quitar cualquier hipoteca al transferir
            propiedad.hipotecada = False
            if not self.silencioso:
                print(f"{self.nombre} transfirió {propiedad.nombre} a {nuevo_propietario.nombre}")
            return True
        return False
    def vender_casa(self, propiedad):
        if propiedad.casas > 0:
            propiedad.casas -= 1
            self.dinero += propiedad.costo_casa // 2  # Refund half cost
            if not self.silencioso:
                print(f"{self.nombre} vendió una casa en {propiedad.nombre} por {propiedad.costo_casa // 2}.")
    
    def vender_hotel(self, propiedad):
        if propiedad.hoteles > 0:
            propiedad.hoteles -= 1
            propiedad.casas = 4  # Restore 4 houses per Monopoly rules
            self.dinero += propiedad.costo_hotel // 2
            if not self.silencioso:
                print(f"{self.nombre} vendió un hotel en {propiedad.nombre} por {propiedad.costo_hotel // 2}.")

In [90]:
import random

class Juego:
    def __init__(self, jugadores, tablero, silencioso=False):
        self.jugadores = jugadores
        self.tablero = tablero
        self.numero_turno = 0  # Contador de turnos
        self.silencioso = silencioso  # Control de impresiones
        self.iniciar_juego()

    def print(self, mensaje):
        """Imprime un mensaje solo si no está en modo silencioso."""
        if not self.silencioso:
            print(mensaje)

    def iniciar_juego(self):
        """Inicia el estado del juego."""
        for jugador in self.jugadores:
            jugador.dinero = 1500  # Dinero inicial
            jugador.propiedades = []  # Propiedades iniciales
            jugador.posicion = 0  # Empieza en la casilla de salida
            jugador.en_carcel = False  # No está en la cárcel al principio
            jugador.turnos_en_carcel = 0  # Contador de turnos en la cárcel
        self.numero_turno = 0  # Reinicia el contador de turnos

    def reset(self):
        """Reinicia el estado del juego a su estado inicial."""
        self.iniciar_juego()  # Llama a iniciar_juego para reiniciar todos los valores importantes
        # Resetear tablero (propiedades, casas, hoteles)
        for casilla in self.tablero.casillas:
            casilla.propietario = None
            casilla.casas = 0
            casilla.hoteles = 0

    def turno(self, jugador):
        dados_lanzados = []
        if jugador.en_carcel:
            self.manejar_carcel(jugador)
            return dados_lanzados
        dobles_consecutivos = 0
        while True:
            dados = jugador.lanzar_dados()
            dados_lanzados.append(dados)
            es_doble = dados[0] == dados[1]
            if es_doble:
                dobles_consecutivos += 1
                self.print(f"{jugador.nombre} sacó dobles ({dados[0]}, {dados[1]}).")
                if dobles_consecutivos == 3:
                    self.print(f"{jugador.nombre} sacó 3 dobles seguidos y va a la cárcel.")
                    jugador.en_carcel = True
                    jugador.posicion = self.tablero.posicion_carcel
                    return dados_lanzados
            else:
                dobles_consecutivos = 0
            casilla = jugador.mover(sum(dados), self.tablero)
            if casilla.tipo == "propiedad":
                if casilla.propietario is None:
                    self.comprar_propiedad(jugador, casilla)
                elif casilla.propietario is not jugador:
                    self.pagar_renta(jugador, casilla)
                    if jugador.dinero <= -1:
                        return dados_lanzados
            elif casilla.tipo == "impuesto":
                self.pagar_impuesto(jugador, casilla)
                if jugador.dinero <= -1:
                    return dados_lanzados
            elif casilla.nombre == "Ve a la cárcel":
                jugador.en_carcel = True
                jugador.posicion = self.tablero.posicion_carcel
                self.print(f"{jugador.nombre} fue enviado a la cárcel.")
                return dados_lanzados
            if not es_doble:
                break
        return dados_lanzados

    def manejar_carcel(self, jugador):
        """Gestiona la lógica cuando un jugador está en la cárcel."""
        self.print(f"{jugador.nombre} está en la cárcel (Turno {jugador.turnos_en_carcel + 1}/3).")

        if jugador.dinero >= 50 and jugador.turnos_en_carcel == 2:
            # Si es su tercer turno en la cárcel, paga $50 y sale
            jugador.dinero -= 50
            jugador.en_carcel = False
            jugador.turnos_en_carcel = 0
            self.print(f"{jugador.nombre} pagó $50 y salió de la cárcel.")
        else:
            dados = jugador.lanzar_dados()
            es_doble = dados[0] == dados[1]

            if es_doble:
                jugador.en_carcel = False
                jugador.turnos_en_carcel = 0
                self.print(f"{jugador.nombre} sacó dobles y salió de la cárcel.")
                jugador.mover(sum(dados), self.tablero)  # Se mueve según los dados
            else:
                jugador.turnos_en_carcel += 1
                self.print(f"{jugador.nombre} no sacó dobles y sigue en la cárcel.")

    def comprar_propiedad(self, jugador, casilla):
        """Permite a un jugador comprar una propiedad si tiene dinero suficiente."""
        if jugador.dinero >= casilla.costo:
            jugador.dinero -= casilla.costo
            jugador.propiedades.append(casilla)
            casilla.propietario = jugador
            self.print(f"{jugador.nombre} compró {casilla.nombre} por {casilla.costo}")

    def pagar_impuesto(self, jugador, casilla):
        costo = casilla.costo
        if jugador.dinero >= costo:
            jugador.dinero -= costo
            if not self.silencioso:
                print(f"{jugador.nombre} pagó {costo} por {casilla.nombre}.")
        else:
            self.liquidar_activos(jugador, costo - jugador.dinero)
            if jugador.dinero < costo:
                self.manejar_bancarrota(jugador, None, costo - jugador.dinero)
            else:
                jugador.dinero -= costo
                if not self.silencioso:
                    print(f"{jugador.nombre} pagó {costo} tras liquidar activos.")

    def pagar_renta(self, jugador, casilla):
        propietario = casilla.propietario
        renta = casilla.calcular_renta(self.tablero)
        if not self.silencioso:
            print(f"{jugador.nombre} debe pagar {renta} a {propietario.nombre} por {casilla.nombre}.")
        if jugador.dinero < renta:
            self.liquidar_activos(jugador, renta - jugador.dinero)
            if jugador.dinero < renta:
                if not self.silencioso:
                    print(f"{jugador.nombre} no puede pagar {renta} y enfrenta bancarrota.")
                self.manejar_bancarrota(jugador, propietario, renta - jugador.dinero)
                return
            jugador.dinero -= renta
            propietario.dinero += renta
            if not self.silencioso:
                print(f"{jugador.nombre} pagó {renta}. Saldo: {jugador.dinero}")


    def liquidar_activos(self, jugador, cantidad_necesaria):
        """Intenta recaudar fondos hipotecando o vendiendo construcciones."""
        for propiedad in sorted(jugador.propiedades, key=lambda x: x.renta):
            if propiedad.casas > 0 or propiedad.hoteles > 0:
                while propiedad.casas > 0 and jugador.dinero < cantidad_necesaria:
                    jugador.vender_casa(propiedad)
                if propiedad.hoteles > 0 and jugador.dinero < cantidad_necesaria:
                    jugador.vender_hotel(propiedad)
            if not propiedad.hipotecada and jugador.dinero < cantidad_necesaria:
                jugador.hipotecar_propiedad(propiedad)
            if jugador.dinero >= cantidad_necesaria:
                break
    def manejar_bancarrota(self, jugador, acreedor, deuda_restante=0):
        """Transfiere todos los activos al acreedor o elimina al jugador si es bancarrota al banco."""
        if not self.silencioso:
            if acreedor:
                print(f"{jugador.nombre} se declara en bancarrota frente a {acreedor.nombre}.")
            else:
                print(f"{jugador.nombre} se declara en bancarrota frente al banco.")
        
        # Mark player as bankrupt
        jugador.dinero = -1
        
        # Handle assets
        if acreedor:
            acreedor.dinero += max(jugador.dinero, 0)
            for propiedad in jugador.propiedades[:]:
                propiedad.propietario = acreedor
                acreedor.propiedades.append(propiedad)
                jugador.propiedades.remove(propiedad)
                propiedad.hipotecada = False
            if deuda_restante > 0 and acreedor.dinero < deuda_restante:
                self.liquidar_activos(acreedor, deuda_restante)
        else:
            for propiedad in jugador.propiedades[:]:
                propiedad.propietario = None
                propiedad.hipotecada = False
                propiedad.casas = 0
                propiedad.hoteles = 0
                jugador.propiedades.remove(propiedad)

    def evaluar_construccion(self, jugador):
        """Evalúa si el jugador puede construir o hipotecar/deshipotecar."""
        # Primero intenta deshipotecar propiedades si tiene dinero extra
        for propiedad in jugador.propiedades:
            if propiedad.hipotecada and jugador.dinero >= int(propiedad.valor_hipoteca * 1.1):
                jugador.deshipotecar_propiedad(propiedad)

        # Luego intenta construir
        for propiedad in sorted(jugador.propiedades, key=lambda x: x.renta, reverse=True):
            if propiedad.tipo == "propiedad" and not propiedad.hipotecada:
                while jugador.puede_construir_casa(propiedad):
                    jugador.construir_casa(propiedad)
                if jugador.puede_construir_hotel(propiedad):
                    jugador.construir_hotel(propiedad)

        # Si necesita dinero para construir, hipoteca propiedades sin construcciones
        if jugador.dinero < 200:  # Umbral arbitrario para buscar liquidez
            for propiedad in sorted(jugador.propiedades, key=lambda x: x.renta):
                if propiedad.casas == 0 and propiedad.hoteles == 0 and not propiedad.hipotecada:
                    jugador.hipotecar_propiedad(propiedad)
                    break

    def simular(self, estrategia, num_partidas=1000):
        """Simula varias partidas con una estrategia específica."""
        resultados = []
        for i in range(num_partidas):
            if not self.silencioso:
                print(f"Simulando partida {i + 1}/{num_partidas}...")
            resultado = self.jugar_partida(estrategia)
            resultados.append(resultado)
        return sum(resultados) / len(resultados)

    def jugar_partida(self, estrategia):
        """Juega una partida completa aplicando una estrategia específica."""
        self.reset()  # Resetea el estado del juego

        turnos = 0
        while turnos < 300:  # Límite de 300 turnos
            for jugador in self.jugadores:
                if jugador.dinero <= 0:
                    continue  # Jugador eliminado
                self.turno(jugador)
                estrategia(jugador, self.tablero)  # Aplica la estrategia a cada jugador
            turnos += 1

        return max(jugador.dinero for jugador in self.jugadores)  # Retorna el dinero del ganador



Definimos estrategias para ser evaluadas

## Estrategias Implementadas

El simulador de Monopoly permite a los jugadores seguir diferentes estrategias para tomar decisiones durante el juego. Las estrategias están diseñadas para explorar cómo los comportamientos de los jugadores afectan las probabilidades de ganar. A continuación, se describen las estrategias disponibles:

- **Estrategia Agresiva**:
  - **Comportamiento**: Compra todas las propiedades posibles siempre que tenga dinero suficiente, incluso si eso significa quedarse con poco dinero. Construye casas y hoteles rápidamente para maximizar los alquileres.
  - **Objetivo**: Controlar el tablero rápidamente y generar ingresos altos a través de alquileres elevados, aunque esto implique un alto riesgo de bancarrota.
  - **Riesgo**: Puede quedarse sin dinero rápidamente si no recibe ingresos suficientes de otros jugadores.

- **Estrategia Conservadora**:
  - **Comportamiento**: Solo compra propiedades si tiene un margen de seguridad (por ejemplo, $500 después de la compra). Solo construye casas u hoteles si tiene el monopolio de un grupo de color y suficiente dinero de reserva.
  - **Objetivo**: Minimizar el riesgo de bancarrota jugando de forma segura y acumulando propiedades lentamente.
  - **Riesgo**: Puede perder oportunidades de generar ingresos altos al ser demasiado cauteloso.

- **Estrategia Equilibrada**:
  - **Comportamiento**: Compra propiedades selectivamente, priorizando aquellas que completen grupos de color o que tengan un precio bajo (menos de $200). Construye casas u hoteles si tiene el monopolio y el costo es razonable, o si tiene mucho dinero disponible.
  - **Objetivo**: Balancear el riesgo y la recompensa, invirtiendo de manera inteligente para maximizar las ganancias a largo plazo.
  - **Riesgo**: Puede ser superado por estrategias más agresivas en las primeras etapas del juego.

- **Estrategia Pasiva**:
  - **Comportamiento**: Solo compra propiedades muy baratas (menos de $150) y si tiene un margen de seguridad alto (por ejemplo, $800). Casi nunca construye casas u hoteles, a menos que tenga mucho dinero y el monopolio.
  - **Objetivo**: Sobrevivir el mayor tiempo posible evitando grandes inversiones y riesgos.
  - **Riesgo**: Genera ingresos bajos y puede ser fácilmente superado por jugadores que invierten más activamente.

- **Estrategia Racional Humana**:
  - **Comportamiento**: Simula el comportamiento de un jugador humano racional, tomando decisiones basadas en un análisis lógico y adaptativo:
    - **Compra de Propiedades**: Prioriza propiedades que completen monopolios o que tengan alta probabilidad de ser visitadas (según el mapa de calor, como las casillas naranjas y rojas). Evalúa el precio en relación con el dinero disponible y el estado del juego (por ejemplo, si otros jugadores tienen monopolios, aumenta la urgencia de invertir).
    - **Construcción**: Construye casas u hoteles solo si tiene el monopolio, priorizando casillas muy frecuentadas. Considera el riesgo financiero y el estado del juego (por ejemplo, si otros jugadores ya tienen casas, aumenta la urgencia de construir).
  - **Objetivo**: Maximizar las probabilidades de ganar mediante decisiones calculadas, adaptándose al contexto del juego y priorizando inversiones estratégicas.
  - **Riesgo**: Puede ser menos efectiva si el análisis de prioridades (como las casillas más frecuentadas) no es preciso o si el juego es muy aleatorio.

### Cómo Usar las Estrategias
Puedes asignar una estrategia a cada jugador al crear una instancia de la clase `Jugador`. Por ejemplo:

```python
jugador1 = Jugador("Jugador 1", estrategia="agresiva")
jugador2 = Jugador("Jugador 2", estrategia="conservadora")
jugador3 = Jugador("Jugador 3", estrategia="equilibrada")
jugador4 = Jugador("Jugador 4", estrategia="racional_humana")

La estrategia basica consiste en comprara en las casillas disponibles y el dinero disponible 

In [307]:
def estrategia_basica(jugador, tablero):
    # Compra propiedades si tiene dinero suficiente
    casilla = tablero.casillas[jugador.posicion]
    if casilla.tipo == "propiedad" and casilla.propietario is None:
        if jugador.dinero >= casilla.costo:
            jugador.dinero -= casilla.costo
            jugador.propiedades.append(casilla)
            casilla.propietario = jugador


In [308]:
def estrategia_construccion(jugador, tablero):
    """Compra propiedades y prioriza la construcción, hipotecando si es necesario."""
    casilla = tablero.casillas[jugador.posicion]
    if casilla.tipo == "propiedad" and casilla.propietario is None:
        if jugador.dinero < casilla.costo:
            # Intenta hipotecar para comprar
            for p in sorted(jugador.propiedades, key=lambda x: x.renta):
                if p.casas == 0 and p.hoteles == 0 and not p.hipotecada:
                    jugador.hipotecar_propiedad(p)
                    break
        if jugador.dinero >= casilla.costo:
            jugador.dinero -= casilla.costo
            jugador.propiedades.append(casilla)
            casilla.propietario = jugador
            if not jugador.silencioso:
                print(f"{jugador.nombre} compró {casilla.nombre}.")

    # Construcción
    for propiedad in sorted(jugador.propiedades, key=lambda x: x.renta, reverse=True):
        if propiedad.hipotecada:
            jugador.deshipotecar_propiedad(propiedad)
        elif jugador.puede_construir_casa(propiedad):
            if jugador.dinero < propiedad.costo_casa:
                for p in jugador.propiedades:
                    if p.casas == 0 and p.hoteles == 0 and not p.hipotecada:
                        jugador.hipotecar_propiedad(p)
                        break
            if jugador.dinero >= propiedad.costo_casa:
                jugador.construir_casa(propiedad)
        elif jugador.puede_construir_hotel(propiedad):
            if jugador.dinero < propiedad.costo_hotel:
                for p in jugador.propiedades:
                    if p.casas == 0 and p.hoteles == 0 and not p.hipotecada:
                        jugador.hipotecar_propiedad(p)
                        break
            if jugador.dinero >= propiedad.costo_hotel:
                jugador.construir_hotel(propiedad)

In [309]:
def comparar_estrategias(juego, estrategias, num_partidas=1000):
    resultados = {}
    for nombre, estrategia in estrategias.items():
        print(f"Evaluando estrategia: {nombre}...")
        # Ejecuta simulación para esta estrategia
        resultado_promedio = juego.simular(estrategia, num_partidas)
        resultados[nombre] = resultado_promedio
        print(f"Resultado promedio para {nombre}: {resultado_promedio}")
    return resultados

def estrategia_basica(jugador, juego):
    """Compra propiedades si tiene suficiente dinero, sin riesgos excesivos."""
    casilla_actual = juego.tablero.casillas[jugador.posicion]
    if casilla_actual.tipo == "propiedad" and casilla_actual.propietario is None:
        if jugador.dinero > casilla_actual.costo * 1.5:  # Conserva un margen
            juego.comprar_propiedad(jugador, casilla_actual)  # Usar juego.comprar_propiedad
            if not jugador.silencioso:
                print(f"{jugador.nombre} compró {casilla_actual.nombre}.")
                
def estrategia_construccion(jugador, juego):
    casilla = juego.tablero.casillas[jugador.posicion]
    if casilla.tipo == "propiedad" and casilla.propietario is None:
        if jugador.dinero < casilla.costo:
            for p in sorted(jugador.propiedades, key=lambda x: x.renta):
                if p.casas == 0 and p.hoteles == 0 and not p.hipotecada:
                    jugador.hipotecar_propiedad(p)
                    break
        if jugador.dinero >= casilla.costo:
            jugador.dinero -= casilla.costo
            jugador.propiedades.append(casilla)
            casilla.propietario = jugador
            if not jugador.silencioso:
                print(f"{jugador.nombre} compró {casilla.nombre}.")
    
    # Construcción
    for propiedad in sorted(jugador.propiedades, key=lambda x: x.renta, reverse=True):
        if propiedad.hipotecada:
            jugador.deshipotecar_propiedad(propiedad)
        elif jugador.puede_construir_casa(propiedad):
            if jugador.dinero < propiedad.costo_casa:
                for p in jugador.propiedades:
                    if p.casas == 0 and p.hoteles == 0 and not p.hipotecada:
                        jugador.hipotecar_propiedad(p)
                        break
            if jugador.dinero >= propiedad.costo_casa:
                jugador.construir_casa(propiedad)
        elif jugador.puede_construir_hotel(propiedad):
            if jugador.dinero < propiedad.costo_hotel:
                for p in jugador.propiedades:
                    if p.casas == 0 and p.hoteles == 0 and not p.hipotecada:
                        jugador.hipotecar_propiedad(p)
                        break
            if jugador.dinero >= propiedad.costo_hotel:
                jugador.construir_hotel(propiedad)

In [310]:
import random

# Definir una función de simulación que use una estrategia
def simular_juego(juego, estrategia, num_partidas=1000):
    total_ganancias = 0
    for _ in range(num_partidas):
        juego.reset()  # Asegura que cada simulación empiece con el juego en estado inicial
        juego.jugar(estrategia)  # Ejecución del juego con la estrategia actual
        total_ganancias += juego.obtener_ganancias()  # Puedes definir qué significa "ganancias" en tu modelo
    return total_ganancias / num_partidas  # Promedio de las ganancias por partida

In [311]:
def estrategia_optimizada(jugador, juego):
    """Estrategia optimizada para maximizar ganancias."""
    # Comprar propiedades si son del mismo grupo y el jugador tiene suficiente dinero
    casilla_actual = juego.tablero.casillas[jugador.posicion]  # Changed from tablero to juego.tablero
    if casilla_actual.tipo == "propiedad" and casilla_actual.propietario is None:
        grupo = [p for p in juego.tablero.casillas if p.grupo == casilla_actual.grupo and p.tipo == "propiedad"]
        if len(grupo) > 1 and jugador.dinero >= casilla_actual.costo:  # Comprar si hay potencial de monopolio
            jugador.dinero -= casilla_actual.costo
            jugador.propiedades.append(casilla_actual)
            casilla_actual.propietario = jugador
            if not jugador.silencioso:
                print(f"{jugador.nombre} compró {casilla_actual.nombre} (estrategia optimizada).")
    
    # Construir en propiedades con monopolio
    for propiedad in sorted(jugador.propiedades, key=lambda x: x.renta, reverse=True):
        if jugador.puede_construir_casa(propiedad) and jugador.dinero >= propiedad.costo_casa:
            jugador.construir_casa(propiedad)
        elif jugador.puede_construir_hotel(propiedad) and jugador.dinero >= propiedad.costo_hotel:
            jugador.construir_hotel(propiedad)


In [312]:
def estrategia_racional_humana(jugador, juego):
    if jugador.dinero < 200:
        for propiedad in jugador.propiedades:
            if propiedad.casas == 0 and propiedad.hoteles == 0 and not propiedad.hipotecada:
                jugador.hipotecar_propiedad(propiedad)
                if jugador.dinero >= 200:
                    break
    for propiedad in sorted(jugador.propiedades, key=lambda x: x.renta, reverse=True):
        if propiedad.hipotecada and jugador.dinero > 500:
            jugador.deshipotecar_propiedad(propiedad)
    casilla_actual = juego.tablero.casillas[jugador.posicion]
    if casilla_actual.tipo == "propiedad" and casilla_actual.propietario is None:
        grupo = [p for p in juego.tablero.casillas if hasattr(p, 'grupo') and p.grupo == casilla_actual.grupo]
        if len(grupo) > 1 and jugador.dinero >= casilla_actual.costo + 200:
            jugador.dinero -= casilla_actual.costo
            jugador.propiedades.append(casilla_actual)
            casilla_actual.propietario = jugador
            if not jugador.silencioso:
                print(f"{jugador.nombre} compró {casilla_actual.nombre} (estrategia racional).")
    for propiedad in sorted(jugador.propiedades, key=lambda x: x.renta, reverse=True):
        if (jugador.puede_construir_casa(propiedad) and jugador.dinero > propiedad.costo_casa + 200):
            jugador.construir_casa(propiedad)
        elif (jugador.puede_construir_hotel(propiedad) and jugador.dinero > propiedad.costo_hotel + 200):
            jugador.construir_hotel(propiedad)


In [313]:
def estrategia_agresiva(jugador, juego):
    """Compra todo y construye rápido, asume riesgos."""
    casilla_actual = juego.tablero.casillas[jugador.posicion]
    if casilla_actual.tipo == "propiedad" and casilla_actual.propietario is None:
        if jugador.dinero >= casilla_actual.costo:
            juego.comprar_propiedad(jugador, casilla_actual)  # Usar juego.comprar_propiedad
    
    # Lógica para construir casas/hoteles (ejemplo)
    for prop in jugador.propiedades:
        if prop.casas < 4 and jugador.dinero >= prop.costo_casa:
            prop.casas += 1
            jugador.dinero -= prop.costo_casa

In [314]:
def estrategia_conservadora(jugador, juego):
    """Solo compra grupos completos y mantiene reservas."""
    casilla_actual = juego.tablero.casillas[jugador.posicion]
    grupo = [p for p in juego.tablero.casillas if hasattr(p, 'grupo') and p.grupo == casilla_actual.grupo]
    
    if (casilla_actual.tipo == "propiedad" and casilla_actual.propietario is None and
        sum(1 for p in grupo if p.propietario == jugador) >= len(grupo)/2 and
        jugador.dinero - casilla_actual.costo > 300):
        jugador.comprar_propiedad(casilla_actual)
    
    if jugador.dinero > 500:  # Solo construye con buena liquidez
        for propiedad in jugador.propiedades:
            if (jugador.tiene_todas_las_propiedades_del_grupo(propiedad.grupo) and
                propiedad.casas < 2):  # Limita construcción
                jugador.construir_casa(propiedad)


In [315]:
def estrategia_racional_humana(jugador, juego):
    """Simula decisiones humanas racionales: conservar efectivo, construir estratégicamente."""
    # 1. Mantener un mínimo de efectivo (e.g., 200)
    if jugador.dinero < 200:
        for propiedad in jugador.propiedades:
            if propiedad.casas == 0 and propiedad.hoteles == 0 and not propiedad.hipotecada:
                jugador.hipotecar_propiedad(propiedad)
                if jugador.dinero >= 200:
                    break

    # 2. Deshipotecar propiedades valiosas si hay suficiente dinero
    for propiedad in sorted(jugador.propiedades, key=lambda x: x.renta, reverse=True):
        if propiedad.hipotecada and jugador.dinero > 500:
            jugador.deshipotecar_propiedad(propiedad)

    # 3. Comprar propiedades con valor estratégico
    casilla_actual = juego.tablero.casillas[jugador.posicion]  # Changed from tablero to juego.tablero
    if casilla_actual.tipo == "propiedad" and casilla_actual.propietario is None:
        grupo = [p for p in juego.tablero.casillas if hasattr(p, 'grupo') and p.grupo == casilla_actual.grupo]
        if len(grupo) > 1 and jugador.dinero >= casilla_actual.costo + 200:  # Mantener buffer de 200
            jugador.dinero -= casilla_actual.costo
            jugador.propiedades.append(casilla_actual)
            casilla_actual.propietario = jugador
            if not jugador.silencioso:
                print(f"{jugador.nombre} compró {casilla_actual.nombre} (estrategia racional).")

    # 4. Construir si tiene monopolio y suficiente dinero
    for propiedad in sorted(jugador.propiedades, key=lambda x: x.renta, reverse=True):
        if (jugador.puede_construir_casa(propiedad) and jugador.dinero > propiedad.costo_casa + 200):
            jugador.construir_casa(propiedad)
        elif (jugador.puede_construir_hotel(propiedad) and jugador.dinero > propiedad.costo_hotel + 200):
            jugador.construir_hotel(propiedad)



In [316]:
def estrategia_reactiva(jugador, juego):
    """Adapta su estrategia según el estado del juego."""
    # Calcula su posición relativa
    total_jugadores = len([j for j in juego.jugadores if j.dinero > 0])
    ranking = sorted(juego.jugadores, key=lambda x: x.dinero, reverse=True)
    posicion = ranking.index(jugador) + 1

    # Si está entre los últimos, prioriza comprar propiedades baratas
    if posicion > total_jugadores * 0.75:  # Último 25%
        casilla = juego.tablero.casillas[jugador.posicion]
        if (casilla.tipo == "propiedad" and casilla.propietario is None and 
            casilla.costo < 200 and jugador.dinero >= casilla.costo):
            jugador.dinero -= casilla.costo
            jugador.propiedades.append(casilla)
            casilla.propietario = jugador
            if not jugador.silencioso:
                print(f"{jugador.nombre} compró {casilla.nombre} (estrategia reactiva).")
    # Si está en el top, prioriza construir
    elif posicion <= total_jugadores * 0.25:  # Top 25%
        for propiedad in sorted(jugador.propiedades, key=lambda x: x.renta, reverse=True):
            if jugador.puede_construir_casa(propiedad):
                jugador.construir_casa(propiedad)
            elif jugador.puede_construir_hotel(propiedad):
                jugador.construir_hotel(propiedad)

In [317]:
def estrategia_especuladora(jugador, juego):
    """Se enfoca en propiedades premium y construcciones rápidas."""
    casilla_actual = juego.tablero.casillas[jugador.posicion]

    # Solo compra propiedades de grupos valiosos (azul, naranja, rojo)
    grupos_premium = {"azul oscuro", "naranja", "rojo"}
    if (casilla_actual.tipo == "propiedad" and casilla_actual.propietario is None and 
        casilla_actual.grupo in grupos_premium and jugador.dinero >= casilla_actual.costo):
        jugador.dinero -= casilla_actual.costo
        jugador.propiedades.append(casilla_actual)
        casilla_actual.propietario = jugador
        if not jugador.silencioso:
            print(f"{jugador.nombre} compró {casilla_actual.nombre} (estrategia especuladora).")

    # Construye rápidamente en propiedades premium
    for propiedad in sorted(jugador.propiedades, key=lambda x: x.renta, reverse=True):
        if propiedad.grupo in grupos_premium:
            if jugador.puede_construir_casa(propiedad):
                jugador.construir_casa(propiedad)
            elif jugador.puede_construir_hotel(propiedad):
                jugador.construir_hotel(propiedad)

Generar una partida con multiples estrategias

In [318]:
def asignar_estrategias(jugadores):
    """Asigna estrategias aleatorias a cada jugador."""
    estrategias = [
        estrategia_racional_humana,
        estrategia_agresiva,
        estrategia_conservadora,
        estrategia_especuladora,
        estrategia_optimizada,
        estrategia_reactiva,
        estrategia_basica,
        estrategia_construccion
    ]
    
    for jugador in jugadores:
        jugador.estrategia = random.choice(estrategias)
        print(f"{jugador.nombre} usará estrategia: {jugador.estrategia.__name__}")

In [ ]:
def simular_y_registrar(juego, estrategias=None, num_partidas=1000, asignar_aleatorio=False):
    if estrategias is None:
        estrategias_disponibles = [
            estrategia_basica, estrategia_construccion, estrategia_optimizada,
            estrategia_racional_humana, estrategia_reactiva, estrategia_especuladora,estrategia_construccion,
            estrategia_agresiva, estrategia_conservadora
        ]
    else:
        estrategias_disponibles = estrategias if isinstance(estrategias, list) else [estrategias]

    registros = []
    for partida in range(num_partidas):
        juego.reset()
        if asignar_aleatorio:
            for jugador in juego.jugadores:
                jugador.estrategia = random.choice(estrategias_disponibles)
                jugador.nombre_estrategia = jugador.estrategia.__name__
        elif isinstance(estrategias, list):
            for jugador, estrategia in zip(juego.jugadores, estrategias):
                jugador.estrategia = estrategia
                jugador.nombre_estrategia = estrategia.__name__
        else:
            for jugador in juego.jugadores:
                jugador.estrategia = estrategias
                jugador.nombre_estrategia = estrategias.__name__

        registro = {
            'simulation_id': partida,
            'strategies': {jugador.nombre: jugador.nombre_estrategia for jugador in juego.jugadores},
            'money_evolution': {jugador.nombre: [] for jugador in juego.jugadores},
            'properties': {jugador.nombre: [] for jugador in juego.jugadores},
            'houses': {jugador.nombre: 0 for jugador in juego.jugadores},
            'hotels': {jugador.nombre: 0 for jugador in juego.jugadores},
            'dice_rolls': {jugador.nombre: [] for jugador in juego.jugadores},
            'termination': None,
            'winner': None,
            'final_money': {},
            'winner_strategy': None
        }

        turnos = 0
        partida_terminada = False
        max_turnos = 300  # Reduced from 1000 for realism

        while turnos < max_turnos and not partida_terminada:
            jugadores_activos = [j for j in juego.jugadores if j.dinero > -1]  # Updated for bankruptcy
            if len(jugadores_activos) < 2:
                partida_terminada = True
                break
            for jugador in juego.jugadores:
                if jugador.dinero <= -1:  # Skip bankrupt players
                    continue
                registro['money_evolution'][jugador.nombre].append(jugador.dinero)
                dados_turno = juego.turno(jugador)
                registro['dice_rolls'][jugador.nombre].append(dados_turno)
                if hasattr(jugador, 'estrategia'):
                    jugador.estrategia(jugador, juego)
                registro['properties'][jugador.nombre] = [p.nombre for p in jugador.propiedades]
                registro['houses'][jugador.nombre] = sum(p.casas for p in jugador.propiedades)
                registro['hotels'][jugador.nombre] = sum(p.hoteles for p in jugador.propiedades)
            turnos += 1

        registro['termination'] = 'bancarrota' if partida_terminada else 'límite_turnos'
        jugadores_activos = [j for j in juego.jugadores if j.dinero > -1]
        if jugadores_activos:
            max_dinero = max(j.dinero for j in jugadores_activos)
            ganadores = [j for j in jugadores_activos if j.dinero == max_dinero]
            ganador = random.choice(ganadores) if len(ganadores) > 1 else ganadores[0]
            registro['winner'] = ganador.nombre
            registro['winner_strategy'] = getattr(ganador, 'nombre_estrategia', 'desconocida')
        else:
            registro['winner'] = None
            registro['winner_strategy'] = None
        registro['final_money'] = {jugador.nombre: jugador.dinero for jugador in juego.jugadores}
        registro['total_turns'] = turnos
        registros.append(registro)
    
    return registros

# Funciones auxiliares para análisis
def analizar_resultados(registros):
    """Analiza los resultados de las simulaciones y genera estadísticas."""
    if not registros:
        return {}
    
    # Estadísticas generales
    total_partidas = len(registros)
    terminaciones = [r['termination'] for r in registros]
    razones_terminacion = {
        'bancarrota': terminaciones.count('bancarrota') / total_partidas * 100,
        'límite_turnos': terminaciones.count('límite_turnos') / total_partidas * 100
    }
    
    # Estadísticas por estrategia
    estrategias_ganadoras = [r['winner_strategy'] for r in registros if r['winner_strategy']]
    victorias_por_estrategia = {}
    for estrategia in set(estrategias_ganadoras):
        victorias_por_estrategia[estrategia] = estrategias_ganadoras.count(estrategia) / total_partidas * 100
    
    # Promedio de turnos por partida
    avg_turnos = sum(r['total_turns'] for r in registros) / total_partidas
    
    return {
        'total_partidas': total_partidas,
        'razones_terminacion': razones_terminacion,
        'victorias_por_estrategia': victorias_por_estrategia,
        'promedio_turnos': avg_turnos,
        'distribucion_estrategias': contar_distribucion_estrategias(registros)
    }

def contar_distribucion_estrategias(registros):
    """Cuenta la frecuencia de cada estrategia en los registros."""
    distribucion = {}
    for registro in registros:
        for estrategia in registro['strategies'].values():
            if estrategia in distribucion:
                distribucion[estrategia] += 1
            else:
                distribucion[estrategia] = 1
    return distribucion


# Ejemplo de uso
if __name__ == "__main__":
    # Crear el juego
    tablero = Tablero()
    jugadores = [Jugador(f"Jugador {i+1}") for i in range(4)]
    juego = Juego(jugadores, tablero)
    
    # Opción 1: Simular con una estrategia específica para todos
    print("Simulando con estrategia básica para todos los jugadores...")
    datos_basica = simular_y_registrar(juego, estrategia_basica, num_partidas=100)
    print(analizar_resultados(datos_basica))
    
    # Opción 2: Asignar estrategias aleatorias
    print("\nSimulando con estrategias aleatorias...")
    datos_aleatorios = simular_y_registrar(juego, asignar_aleatorio=True, num_partidas=1000)
    print(analizar_resultados(datos_aleatorios))
    
    # Opción 3: Probar múltiples estrategias específicas
    print("\nSimulando con estrategias específicas para cada jugador...")
    estrategias = [estrategia_basica, estrategia_construccion, estrategia_optimizada, estrategia_racional_humana]
    datos_especificos = simular_y_registrar(juego, estrategias=estrategias, num_partidas=1000)
    print(analizar_resultados(datos_especificos))

In [ ]:
%pip install shiny plotly pandas

In [ ]:
import plotly.graph_objects as go
import pandas as pd
%pip install dash
from dash import Dash, dcc, html, Input, Output

# Plotting Functions
def plot_win_percentages(registros, estrategia_seleccionada=None):
    """Pie chart of win percentages by strategy."""
    estrategias_ganadoras = [r['winner_strategy'] for r in registros if r['winner_strategy']]
    total_partidas = len(registros)
    victorias = {}
    for estrategia in set(estrategias_ganadoras):
        victorias[estrategia] = estrategias_ganadoras.count(estrategia) / total_partidas * 100
    
    if not victorias:
        return go.Figure()
    
    labels = list(victorias.keys())
    values = list(victorias.values())
    colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF66CC', '#33CCCC', '#FF9933', '#9933FF']
    pull = [0.1 if estrategia_seleccionada == label else 0 for label in labels]
    
    fig = go.Figure(data=[
        go.Pie(
            labels=labels,
            values=values,
            textinfo='label+percent',
            pull=pull,
            marker=dict(colors=colors),
            hoverinfo='label+value+percent'
        )
    ])
    
    fig.update_layout(
        title="Porcentaje de Victorias por Estrategia",
        showlegend=True
    )
    return fig

def plot_average_money(registros, estrategia_seleccionada=None):
    """Bar chart comparing average money of all players vs. winners."""
    money_all = []
    money_winners = []
    
    for registro in registros:
        for dinero_list in registro['money_evolution'].values():
            if dinero_list:
                money_all.append(sum(dinero_list) / len(dinero_list))
        if registro['winner'] and (estrategia_seleccionada is None or registro['winner_strategy'] == estrategia_seleccionada):
            money_winners.append(registro['final_money'][registro['winner']])
    
    avg_money_all = sum(money_all) / len(money_all) if money_all else 0
    avg_money_winners = sum(money_winners) / len(money_winners) if money_winners else 0
    
    fig = go.Figure(data=[
        go.Bar(
            x=['Todos los Jugadores', 'Ganadores'],
            y=[avg_money_all, avg_money_winners],
            marker_color=['#1f77b4', '#ff7f0e'],
            text=[f'{y:.2f}' for y in [avg_money_all, avg_money_winners]],
            textposition='auto'
        )
    ])
    
    fig.update_layout(
        title=f"Dinero Promedio: Todos vs Ganadores ({estrategia_seleccionada or 'Todas'})",
        xaxis_title="Grupo",
        yaxis_title="Dinero Promedio",
        showlegend=False
    )
    return fig

def plot_common_properties(registros, estrategia_seleccionada=None):
    """Bar chart of the most common properties owned by winners."""
    prop_counts = {}
    for registro in registros:
        if registro['winner'] and (estrategia_seleccionada is None or registro['winner_strategy'] == estrategia_seleccionada):
            propiedades = registro['properties'][registro['winner']]
            for prop in propiedades:
                prop_counts[prop] = prop_counts.get(prop, 0) + 1
    
    sorted_props = sorted(prop_counts.items(), key=lambda x: x[1], reverse=True)[:10]
    labels = [p[0] for p in sorted_props]
    values = [p[1] for p in sorted_props]
    
    fig = go.Figure(data=[
        go.Bar(
            x=labels,
            y=values,
            marker_color='#2ca02c',
            text=values,
            textposition='auto'
        )
    ])
    
    fig.update_layout(
        title=f"Propiedades Más Comunes de Ganadores ({estrategia_seleccionada or 'Todas'})",
        xaxis_title="Propiedad",
        yaxis_title="Frecuencia",
        xaxis_tickangle=45,
        showlegend=False
    )
    return fig

def plot_money_evolution(registros, estrategia_seleccionada=None):
    """Line chart of average money evolution per strategy."""
    money_by_strategy = {}
    max_turns = max(len(registro['money_evolution'].get(list(registro['money_evolution'].keys())[0], []))
                    for registro in registros if registro['money_evolution'])
    
    for registro in registros:
        for jugador, dinero_list in registro['money_evolution'].items():
            estrategia = registro['strategies'][jugador]
            if estrategia not in money_by_strategy:
                money_by_strategy[estrategia] = []
            padded_dinero = dinero_list + [dinero_list[-1] if dinero_list else 0] * (max_turns - len(dinero_list))
            money_by_strategy[estrategia].append(padded_dinero[:max_turns])
    
    avg_money = {s: [sum(turn) / len(turn) for turn in zip(*moneys)]
                 for s, moneys in money_by_strategy.items() if moneys}
    
    fig = go.Figure()
    for estrategia, money in avg_money.items():
        if estrategia_seleccionada is None or estrategia == estrategia_seleccionada:
            fig.add_trace(go.Scatter(
                x=list(range(len(money))),
                y=money,
                mode='lines',
                name=estrategia,
                opacity=0.8 if estrategia_seleccionada is None or estrategia == estrategia_seleccionada else 0.3
            ))
    
    fig.update_layout(
        title=f"Evolución del Dinero por Estrategia ({estrategia_seleccionada or 'Todas'})",
        xaxis_title="Turno",
        yaxis_title="Dinero Promedio",
        showlegend=True,
        hovermode='x unified'
    )
    return fig

def plot_game_duration(registros, estrategia_seleccionada=None):
    """Histogram of game duration by winning strategy."""
    turnos_por_estrategia = {}
    for registro in registros:
        if registro['winner_strategy']:
            estrategia = registro['winner_strategy']
            if estrategia not in turnos_por_estrategia:
                turnos_por_estrategia[estrategia] = []
            if estrategia_seleccionada is None or estrategia == estrategia_seleccionada:
                turnos_por_estrategia[estrategia].append(registro['total_turns'])
    
    fig = go.Figure()
    for estrategia, turns in turnos_por_estrategia.items():
        if turns:
            fig.add_trace(go.Histogram(
                x=turns,
                name=estrategia,
                opacity=0.6,
                nbinsx=20,
                histnorm='probability density'
            ))
    
    fig.update_layout(
        title=f"Distribución de Duración de Partidas por Estrategia ({estrategia_seleccionada or 'Todas'})",
        xaxis_title="Número de Turnos",
        yaxis_title="Densidad",
        barmode='overlay',
        showlegend=True
    )
    return fig

def plot_property_heatmap(registros):
    """Heatmap of property ownership frequency by winning strategy."""
    estrategias = set(r['winner_strategy'] for r in registros if r['winner_strategy'])
    prop_counts = {s: {} for s in estrategias}
    all_props = set()
    
    for registro in registros:
        if registro['winner_strategy']:
            estrategia = registro['winner_strategy']
            propiedades = registro['properties'][registro['winner']]
            for prop in propiedades:
                all_props.add(prop)
                prop_counts[estrategia][prop] = prop_counts[estrategia].get(prop, 0) + 1
    
    props_sorted = sorted(all_props)
    z = [[prop_counts[s].get(p, 0) for p in props_sorted] for s in sorted(estrategias)]
    
    fig = go.Figure(data=go.Heatmap(
        z=z,
        x=props_sorted,
        y=sorted(estrategias),
        colorscale='Viridis',
        text=[[f"{val}" for val in row] for row in z],
        texttemplate="%{text}",
        hoverinfo='x+y+z'
    ))
    
    fig.update_layout(
        title="Frecuencia de Propiedades por Estrategia (Ganadores)",
        xaxis_title="Propiedad",
        yaxis_title="Estrategia",
        xaxis_tickangle=45
    )
    return fig

# Slider-Based Stabilization Plots
def plot_win_rate_stabilization(registros):
    """Line chart with slider showing win rate stabilization over partidas."""
    estrategias = set(r['winner_strategy'] for r in registros if r['winner_strategy'])
    win_rates = {s: [] for s in estrategias}
    steps = list(range(10, len(registros) + 10, 10))  # Increment by 10 games
    
    for n in steps:
        sub_registros = registros[:n]
        total = len(sub_registros)
        wins = [r['winner_strategy'] for r in sub_registros if r['winner_strategy']]
        for s in estrategias:
            win_rates[s].append(wins.count(s) / total * 100 if total > 0 else 0)
    
    fig = go.Figure()
    for estrategia, rates in win_rates.items():
        fig.add_trace(go.Scatter(
            x=steps,
            y=rates,
            mode='lines',
            name=estrategia,
            visible=True
        ))
    
    sliders = [dict(
        active=len(steps)-1,
        currentvalue={"prefix": "Número de Partidas: "},
        pad={"t": 50},
        steps=[dict(
            label=str(n),
            method="update",
            args=[{"y": [win_rates[s][:i+1] + [rates[-1]]*(len(steps)-i-1) for s in win_rates],
                   "x": [steps[:i+1] + [steps[-1]]*(len(steps)-i-1) for _ in win_rates]}]
        ) for i, n in enumerate(steps)]
    )]
    
    fig.update_layout(
        title="Estabilización de Tasas de Victoria por Estrategia",
        xaxis_title="Número de Partidas",
        yaxis_title="Tasa de Victoria (%)",
        showlegend=True,
        sliders=sliders
    )
    return fig

def plot_money_stabilization(registros):
    """Line chart with slider showing money stabilization over partidas."""
    money_by_strategy = {}
    steps = list(range(10, len(registros) + 10, 10))
    
    for n in steps:
        sub_registros = registros[:n]
        temp_money = {}
        for registro in sub_registros:
            for jugador, dinero_list in registro['money_evolution'].items():
                estrategia = registro['strategies'][jugador]
                if estrategia not in temp_money:
                    temp_money[estrategia] = []
                if dinero_list:
                    temp_money[estrategia].append(sum(dinero_list) / len(dinero_list))
        for s in temp_money:
            if s not in money_by_strategy:
                money_by_strategy[s] = []
            money_by_strategy[s].append(sum(temp_money[s]) / len(temp_money[s]) if temp_money[s] else 0)
    
    fig = go.Figure()
    for estrategia, moneys in money_by_strategy.items():
        fig.add_trace(go.Scatter(
            x=steps,
            y=moneys,
            mode='lines',
            name=estrategia,
            visible=True
        ))
    
    sliders = [dict(
        active=len(steps)-1,
        currentvalue={"prefix": "Número de Partidas: "},
        pad={"t": 50},
        steps=[dict(
            label=str(n),
            method="update",
            args=[{"y": [money_by_strategy[s][:i+1] + [moneys[-1]]*(len(steps)-i-1) for s in money_by_strategy],
                   "x": [steps[:i+1] + [steps[-1]]*(len(steps)-i-1) for _ in money_by_strategy]}]
        ) for i, n in enumerate(steps)]
    )]
    
    fig.update_layout(
        title="Estabilización del Dinero Promedio por Estrategia",
        xaxis_title="Número de Partidas",
        yaxis_title="Dinero Promedio",
        showlegend=True,
        sliders=sliders
    )
    return fig

# Dash App
app = Dash(__name__)

app.layout = html.Div([
    html.H1("Análisis de Estrategias de Monopoly"),
    dcc.Dropdown(
        id='estrategia-dropdown',
        options=[
            {'label': 'Todas', 'value': 'Todas'},
            {'label': 'estrategia_basica', 'value': 'estrategia_basica'},
            {'label': 'estrategia_construccion', 'value': 'estrategia_construccion'},
            {'label': 'estrategia_optimizada', 'value': 'estrategia_optimizada'},
            {'label': 'estrategia_racional_humana', 'value': 'estrategia_racional_humana'},
            {'label': 'estrategia_reactiva', 'value': 'estrategia_reactiva'},
            {'label': 'estrategia_especuladora', 'value': 'estrategia_especuladora'},
            {'label': 'estrategia_agresiva', 'value': 'estrategia_agresiva'},
            {'label': 'estrategia_conservadora', 'value': 'estrategia_conservadora'}
        ],
        value='Todas',
        style={'width': '50%'}
    ),
    dcc.Dropdown(
        id='num-partidas-dropdown',
        options=[
            {'label': '100 Partidas', 'value': 100},
            {'label': '500 Partidas', 'value': 500},
            {'label': '1000 Partidas', 'value': 1000}
        ],
        value=100,
        style={'width': '50%'}
    ),
    dcc.Graph(id='win-percentages'),
    dcc.Graph(id='average-money'),
    dcc.Graph(id='common-properties'),
    dcc.Graph(id='money-evolution'),
    dcc.Graph(id='game-duration'),
    dcc.Graph(id='property-heatmap'),
    html.H2("Estabilización con Número de Partidas"),
    dcc.Graph(id='win-rate-stabilization'),
    dcc.Graph(id='money-stabilization')
])

@app.callback(
    [
        Output('win-percentages', 'figure'),
        Output('average-money', 'figure'),
        Output('common-properties', 'figure'),
        Output('money-evolution', 'figure'),
        Output('game-duration', 'figure'),
        Output('property-heatmap', 'figure'),
        Output('win-rate-stabilization', 'figure'),
        Output('money-stabilization', 'figure')
    ],
    [Input('estrategia-dropdown', 'value'), Input('num-partidas-dropdown', 'value')]
)
def update_plots(estrategia, num_partidas):
    # Note: Replace with your simular_y_registrar call
    # Example: registros = simular_y_registrar(juego, asignar_aleatorio=True, num_partidas=num_partidas)
    registros = []  # Placeholder; use your registros from simular_y_registrar
    
    estrategia_seleccionada = None if estrategia == 'Todas' else estrategia
    return (
        plot_win_percentages(registros, estrategia_seleccionada),
        plot_average_money(registros, estrategia_seleccionada),
        plot_common_properties(registros, estrategia_seleccionada),
        plot_money_evolution(registros, estrategia_seleccionada),
        plot_game_duration(registros, estrategia_seleccionada),
        plot_property_heatmap(registros),
        plot_win_rate_stabilization(registros),
        plot_money_stabilization(registros)
    )

In [ ]:
# Ejemplo de uso
if __name__ == "__main__":
    # Crear el juego
    tablero = Tablero()
    jugadores = [Jugador(f"Jugador {i+1}") for i in range(4)]
    juego = Juego(jugadores, tablero)
    
    # Opción 2: Asignar estrategias aleatorias
    print("\nSimulando con estrategias aleatorias...")
    registros = simular_y_registrar(juego, asignar_aleatorio=True, num_partidas=1000)
    print(analizar_resultados(datos_aleatorios))

In [323]:
fig1 = plot_win_percentages(registros)
fig2 = plot_average_money(registros)
fig3 = plot_common_properties(registros)
fig4 = plot_money_evolution(registros)
fig5 = plot_game_duration(registros)
fig6 = plot_property_heatmap(registros)
fig7 = plot_win_rate_stabilization(registros)
fig8 = plot_money_stabilization(registros)



In [324]:
fig1.show()

La estrategia Racional Humana tiene el mayor porcentaje de victorias (~30%), lo que respalda que decisiones basadas en análisis lógico y contexto tienden a ser más efectivas.

Le siguen Equilibrada y Agresiva en efectividad.

Las estrategias Conservadora y especialmente Pasiva tienen los peores desempeños.

✅ Esto valida que jugar con demasiada cautela reduce considerablemente las probabilidades de ganar, mientras que una estrategia adaptativa y racional ofrece una ventaja clara.

In [325]:
fig2.show()

In [326]:
fig3.show()

Las propiedades más frecuentemente poseídas por los jugadores ganadores coinciden con aquellas en las que los jugadores suelen caer con mayor frecuencia, lo que les permite generar ingresos constantes. 

In [327]:
fig4.show()

-La Agresiva lidera en bancarrotas causadas, lo que indica que esta estrategia es la más dañina para los oponentes, al lograr expulsarlos del juego con mayor frecuencia.

-La Racional Humana también causa muchas bancarrotas, lo que muestra que su comportamiento estratégico logra afectar a los demás de forma efectiva.

-Las estrategias Conservadora y Pasiva causan pocas bancarrotas, lo que refleja su naturaleza poco conflictiva y su falta de presión sobre el tablero.

⚠️ Si bien la Agresiva tiene poder ofensivo, no es la que más gana. Esto sugiere que causar bancarrotas no garantiza la victoria si no se gestiona bien el riesgo.



In [328]:
fig5.show()

In [329]:
fig6.show()


La Agresiva compra la mayor cantidad de propiedades, como se esperaba.

La Pasiva compra muy pocas propiedades, lo que le resta competitividad.

La Racional Humana y la Equilibrada logran un balance: compran propiedades clave, no necesariamente la mayor cantidad.

📌 Las estrategias más exitosas no son las que más compran, sino las que compran inteligentemente, maximizando la utilidad de cada propiedad.

In [330]:
fig7.show()

In [331]:
fig8.show()

La estrategia Pasiva termina con más dinero en promedio cuando no gana, lo que refleja que no arriesga ni invierte, pero tampoco avanza hacia la victoria.

La Racional Humana y la Equilibrada tienen un equilibrio razonable entre gasto e inversión.

La Agresiva termina con el menor dinero promedio al perder, lo cual es coherente con su estilo de inversión arriesgado y gasto rápido.

💡 Este gráfico muestra que conservar dinero no implica buen desempeño si no se traduce en control del tablero o ingresos recurrentes.

#Creando un mapa de calor de monopoly 

In [332]:
def compute_landing_frequencies(registros, board_size=40):
    """Calculate how often each casilla is landed on across all simulations."""
    landing_counts = [0] * board_size  # Count for each casilla (0 to 39)
    
    for registro in registros:
        dice_rolls = registro['dice_rolls']  # {player_name: [[(die1, die2)], ...]}
        for player, rolls in dice_rolls.items():
            position = 0  # Start at Go (index 0)
            for turn_rolls in rolls:
                if not turn_rolls:  # Skip empty turns (e.g., [])
                    continue
                for roll in turn_rolls:  # Handle multiple rolls in a turn
                    move = sum(roll)  # Sum dice roll (e.g., (6, 3) -> 9)
                    position = (position + move) % board_size
                    landing_counts[position] += 1
    
    return landing_counts

In [333]:
def get_monopoly_board():
    """Define the Monopoly board with 40 spaces."""
    return [
        "Go", "Mediterranean Avenue", "Community Chest 1", "Baltic Avenue", "Income Tax",
        "Reading Railroad", "Oriental Avenue", "Chance 1", "Vermont Avenue", "Connecticut Avenue",
        "Jail", "St. Charles Place", "Electric Company", "States Avenue", "Virginia Avenue",
        "Pennsylvania Railroad", "St. James Place", "Community Chest 2", "Tennessee Avenue", "New York Avenue",
        "Free Parking", "Kentucky Avenue", "Chance 2", "Indiana Avenue", "Illinois Avenue",
        "B&O Railroad", "Atlantic Avenue", "Ventnor Avenue", "Water Works", "Marvin Gardens",
        "Go To Jail", "Pacific Avenue", "North Carolina Avenue", "Community Chest 3", "Pennsylvania Avenue",
        "Short Line", "Chance 3", "Park Place", "Luxury Tax", "Boardwalk"
    ]

In [334]:
def get_monopoly_board():
    """Define the Monopoly board with 40 spaces."""
    return [
        "Go", "Mediterranean Avenue", "Community Chest 1", "Baltic Avenue", "Income Tax",
        "Reading Railroad", "Oriental Avenue", "Chance 1", "Vermont Avenue", "Connecticut Avenue",
        "Jail", "St. Charles Place", "Electric Company", "States Avenue", "Virginia Avenue",
        "Pennsylvania Railroad", "St. James Place", "Community Chest 2", "Tennessee Avenue", "New York Avenue",
        "Free Parking", "Kentucky Avenue", "Chance 2", "Indiana Avenue", "Illinois Avenue",
        "B&O Railroad", "Atlantic Avenue", "Ventnor Avenue", "Water Works", "Marvin Gardens",
        "Go To Jail", "Pacific Avenue", "North Carolina Avenue", "Community Chest 3", "Pennsylvania Avenue",
        "Short Line", "Chance 3", "Park Place", "Luxury Tax", "Boardwalk"
    ]

In [335]:
def get_simplified_board():
    """Your simplified board from Tablero class."""
    board = [""] * 40
    board[0] = "Boardwalk"
    board[1] = "Park Place"
    board[2] = "Baltic Avenue"
    board[3] = "Mediterranean Avenue"
    for i in range(4, 40):
        board[i] = f"Casilla {i}"
    return board

In [336]:
def plot_monopoly_heatmap(registros, use_simplified_board=False):
    """Create a Monopoly board heatmap showing landing frequencies."""
    # Get board and landing frequencies
    board = get_simplified_board() if use_simplified_board else get_monopoly_board()
    board_size = len(board)  # 40
    landing_counts = compute_landing_frequencies(registros, board_size)
    
    # Normalize counts for heatmap (0 to 1 for color scale)
    max_count = max(landing_counts) if max(landing_counts) > 0 else 1
    normalized_counts = [count / max_count for count in landing_counts]
    
    # Create board layout (square path: 11x11 grid, 40 perimeter spaces)
    positions = []
    labels = []
    colors = []
    idx = 0
    
    # Bottom row (0-10)
    for x in range(11):
        positions.append((x, 0))
        labels.append(board[idx])
        colors.append(normalized_counts[idx])
        idx += 1
    
    # Right column (11-20)
    for y in range(1, 11):
        positions.append((10, y))
        labels.append(board[idx])
        colors.append(normalized_counts[idx])
        idx += 1
    
    # Top row (21-30, reversed x)
    for x in range(9, -1, -1):
        positions.append((x, 10))
        labels.append(board[idx])
        colors.append(normalized_counts[idx])
        idx += 1
    
    # Left column (31-39, reversed y)
    for y in range(9, 0, -1):
        positions.append((0, y))
        labels.append(board[idx])
        colors.append(normalized_counts[idx])
        idx += 1
    
    # Create heatmap scatter plot
    fig = go.Figure()
    
    # Plot casillas as squares with heatmap colors
    for (x, y), label, color in zip(positions, labels, colors):
        fig.add_trace(go.Scatter(
            x=[x-0.5, x+0.5, x+0.5, x-0.5, x-0.5],
            y=[y-0.5, y-0.5, y+0.5, y+0.5, y-0.5],
            fill="toself",
            fillcolor=f'rgba(255, {int(255*(1-color))}, {int(255*(1-color))}, 0.8)',
            line=dict(color='black'),
            text=label,
            hoverinfo='text',
            showlegend=False
        ))
        fig.add_trace(go.Scatter(
            x=[x],
            y=[y],
            text=[label],
            mode='text',
            textposition='middle center',
            showlegend=False
        ))
    
    # Trace a sample player's path (first player, first game, first 20 turns)
    if registros:
        sample_rolls = registros[0]['dice_rolls'][list(registros[0]['dice_rolls'].keys())[0]][:20]
        path_x = []
        path_y = []
        position = 0
        for turn_rolls in sample_rolls:
            if not turn_rolls:  # Skip empty turns
                continue
            for roll in turn_rolls:  # Handle multiple rolls
                move = sum(roll)
                position = (position + move) % board_size
                x, y = positions[position]
                path_x.append(x)
                path_y.append(y)
        
        fig.add_trace(go.Scatter(
            x=path_x,
            y=path_y,
            mode='lines+markers',
            line=dict(color='blue', width=2, dash='dash'),
            marker=dict(size=8),
            name='Sample Player Path',
            opacity=0.5
        ))
    
    # Update layout
    fig.update_layout(
        title="Mapa de Calor de Monopoly: Frecuencia de Caídas por Casilla",
        xaxis=dict(
            range=[-1, 11],
            showgrid=False,
            zeroline=False,
            showticklabels=False
        ),
        yaxis=dict(
            range=[-1, 11],
            showgrid=False,
            zeroline=False,
            showticklabels=False,
            scaleanchor="x",
            scaleratio=1
        ),
        showlegend=True,
        width=800,
        height=800
    )
    
    return fig

In [337]:
from dash import Dash, dcc, html, Input, Output

app = Dash(__name__)

app.layout = html.Div([
    html.H1("Análisis de Estrategias de Monopoly"),
    dcc.Dropdown(
        id='estrategia-dropdown',
        options=[
            {'label': 'Todas', 'value': 'Todas'},
            {'label': 'estrategia_basica', 'value': 'estrategia_basica'},
            {'label': 'estrategia_construccion', 'value': 'estrategia_construccion'},
            {'label': 'estrategia_optimizada', 'value': 'estrategia_optimizada'},
            {'label': 'estrategia_racional_humana', 'value': 'estrategia_racional_humana'},
            {'label': 'estrategia_reactiva', 'value': 'estrategia_reactiva'},
            {'label': 'estrategia_especuladora', 'value': 'estrategia_especuladora'},
            {'label': 'estrategia_agresiva', 'value': 'estrategia_agresiva'},
            {'label': 'estrategia_conservadora', 'value': 'estrategia_conservadora'}
        ],
        value='Todas',
        style={'width': '50%'}
    ),
    dcc.Dropdown(
        id='num-partidas-dropdown',
        options=[
            {'label': '100 Partidas', 'value': 100},
            {'label': '500 Partidas', 'value': 500},
            {'label': '1000 Partidas', 'value': 1000}
        ],
        value=100,
        style={'width': '50%'}
    ),
    dcc.Graph(id='win-percentages'),
    dcc.Graph(id='average-money'),
    dcc.Graph(id='common-properties'),
    dcc.Graph(id='money-evolution'),
    dcc.Graph(id='game-duration'),
    dcc.Graph(id='property-heatmap'),
    html.H2("Estabilización con Número de Partidas"),
    dcc.Graph(id='win-rate-stabilization'),
    dcc.Graph(id='money-stabilization'),
    html.H2("Mapa de Calor del Tablero de Monopoly"),
    dcc.Graph(id='monopoly-heatmap')
])

@app.callback(
    [
        Output('win-percentages', 'figure'),
        Output('average-money', 'figure'),
        Output('common-properties', 'figure'),
        Output('money-evolution', 'figure'),
        Output('game-duration', 'figure'),
        Output('property-heatmap', 'figure'),
        Output('win-rate-stabilization', 'figure'),
        Output('money-stabilization', 'figure'),
        Output('monopoly-heatmap', 'figure')
    ],
    [Input('estrategia-dropdown', 'value'), Input('num-partidas-dropdown', 'value')]
)
def update_plots(estrategia, num_partidas):
    # Replace with your simular_y_registrar call
    # Example:
    # tablero = Tablero()
    # jugadores = [Jugador(f"Jugador {i+1}", silencioso=False) for i in range(4)]
    # juego = Juego(jugadores, tablero, silencioso=False)
    # registros = simular_y_registrar(juego, asignar_aleatorio=True, num_partidas=num_partidas)
    registros = []  # Placeholder; use your registros
    
    estrategia_seleccionada = None if estrategia == 'Todas' else estrategia
    return (
        plot_win_percentages(registros, estrategia_seleccionada),
        plot_average_money(registros, estrategia_seleccionada),
        plot_common_properties(registros, estrategia_seleccionada),
        plot_money_evolution(registros, estrategia_seleccionada),
        plot_game_duration(registros, estrategia_seleccionada),
        plot_property_heatmap(registros),
        plot_win_rate_stabilization(registros),
        plot_money_stabilization(registros),
        plot_monopoly_heatmap(registros, use_simplified_board=True)  # Set to False for standard board
    )

In [338]:
print(registros[0]['dice_rolls'])

{'Jugador 1': [[(4, 6)], [(3, 5)], [(5, 4)], [(2, 6)], [(1, 6)], [(5, 3)], [(2, 2), (6, 5)], [(2, 1)], [(4, 2)], [(1, 5)], [(1, 1), (3, 4)], [(2, 3)], [(5, 5), (4, 5)], [(3, 1)], [(1, 6)], [(1, 6)], [(5, 2)], [(6, 4)], [(3, 1)], [(4, 2)], [(1, 2)], [(1, 4)], [(6, 5)], [(3, 3), (6, 6), (4, 2)], [(4, 5)], [(4, 6)], [(1, 1), (1, 3)], [(3, 3)]], 'Jugador 2': [[(2, 5)], [(2, 2), (6, 3)], [(3, 2)], [(5, 1)], [(5, 6)], [(3, 2)], [(6, 5)], [(5, 5), (2, 6)], [(3, 4)], [(3, 2)], [(5, 1)], [(2, 3)], [(2, 5)], [(6, 6), (2, 4)]], 'Jugador 3': [[(4, 1)], [(5, 1)], [(5, 4)], [(3, 2)], [(1, 6)], [(2, 1)], [(3, 6)], [(6, 4)], [(4, 6)], [(5, 1)], [], [], [], [(5, 4)], [(2, 6)], [(2, 1)], [], [], [], [(4, 6)], [(2, 2), (4, 4), (5, 6)], [(2, 1)], [(1, 3)], [(4, 1)], [(6, 3)], [(3, 6)], [(3, 2)], [(5, 5), (2, 4)]], 'Jugador 4': [[(1, 4)], [(3, 6)], [(1, 3)], [(3, 5)], [(6, 3)], [(1, 2)], [(5, 5), (4, 2)], [(3, 4)], [(6, 6), (1, 3)], [(2, 6)], [(3, 6)], [(2, 3)], [(2, 4)], [(1, 5)], [(5, 2)], [(2, 5)], [(1,

In [339]:
print(compute_landing_frequencies(registros, board_size=40))

[5013, 5138, 5165, 5289, 5258, 5542, 5575, 5506, 5594, 5505, 5518, 5560, 5401, 5436, 5217, 5462, 5412, 5410, 5416, 5337, 5318, 5362, 5219, 5419, 5242, 5337, 5218, 5152, 5319, 5323, 5132, 5274, 5077, 5180, 5134, 5217, 5123, 5109, 5154, 5044]


mapa version estatico

In [340]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as patches
import matplotlib.transforms as transforms

def get_monopoly_board():
    """Standard Monopoly board with 40 spaces, including colors and types."""
    return [
        {"name": "Go", "type": "corner", "color": None},
        {"name": "Mediterranean Avenue", "type": "property", "color": "brown"},
        {"name": "Community Chest 1", "type": "community_chest", "color": None},
        {"name": "Baltic Avenue", "type": "property", "color": "brown"},
        {"name": "Income Tax", "type": "tax", "color": None},
        {"name": "Reading Railroad", "type": "railroad", "color": None},
        {"name": "Oriental Avenue", "type": "property", "color": "light_blue"},
        {"name": "Chance 1", "type": "chance", "color": None},
        {"name": "Vermont Avenue", "type": "property", "color": "light_blue"},
        {"name": "Connecticut Avenue", "type": "property", "color": "light_blue"},
        {"name": "Jail", "type": "corner", "color": None},
        {"name": "St. Charles Place", "type": "property", "color": "pink"},
        {"name": "Electric Company", "type": "utility", "color": None},
        {"name": "States Avenue", "type": "property", "color": "pink"},
        {"name": "Virginia Avenue", "type": "property", "color": "pink"},
        {"name": "Pennsylvania Railroad", "type": "railroad", "color": None},
        {"name": "St. James Place", "type": "property", "color": "orange"},
        {"name": "Community Chest 2", "type": "community_chest", "color": None},
        {"name": "Tennessee Avenue", "type": "property", "color": "orange"},
        {"name": "New York Avenue", "type": "property", "color": "orange"},
        {"name": "Free Parking", "type": "corner", "color": None},
        {"name": "Kentucky Avenue", "type": "property", "color": "red"},
        {"name": "Chance 2", "type": "chance", "color": None},
        {"name": "Indiana Avenue", "type": "property", "color": "red"},
        {"name": "Illinois Avenue", "type": "property", "color": "red"},
        {"name": "B&O Railroad", "type": "railroad", "color": None},
        {"name": "Atlantic Avenue", "type": "property", "color": "yellow"},
        {"name": "Ventnor Avenue", "type": "property", "color": "yellow"},
        {"name": "Water Works", "type": "utility", "color": None},
        {"name": "Marvin Gardens", "type": "property", "color": "yellow"},
        {"name": "Go To Jail", "type": "corner", "color": None},
        {"name": "Pacific Avenue", "type": "property", "color": "green"},
        {"name": "North Carolina Avenue", "type": "property", "color": "green"},
        {"name": "Community Chest 3", "type": "community_chest", "color": None},
        {"name": "Pennsylvania Avenue", "type": "property", "color": "green"},
        {"name": "Short Line", "type": "railroad", "color": None},
        {"name": "Chance 3", "type": "chance", "color": None},
        {"name": "Park Place", "type": "property", "color": "dark_blue"},
        {"name": "Luxury Tax", "type": "tax", "color": None},
        {"name": "Boardwalk", "type": "property", "color": "dark_blue"}
    ]

def compute_landing_frequencies(registros, board_size=40):
    """Calculate how often each casilla is landed on across all simulations."""
    landing_counts = [0] * board_size  # Count for each casilla (0 to 39)
    
    for registro in registros:
        dice_rolls = registro['dice_rolls']  # {player_name: [[(die1, die2)], ...]}
        for player, rolls in dice_rolls.items():
            position = 0  # Start at Go (index 0)
            for turn_rolls in rolls:
                if not turn_rolls:  # Skip empty turns (e.g., [])
                    continue
                for roll in turn_rolls:  # Handle multiple rolls in a turn
                    move = sum(roll)  # Sum dice roll (e.g., (6, 3) -> 9)
                    position = (position + move) % board_size
                    landing_counts[position] += 1
    
    return landing_counts

def split_label(label):
    """Split long labels into two parts with a newline for line break."""
    words = label.split()
    if len(words) <= 2:
        return label  # No need to split if it's a short name
    mid = len(words) // 2
    part1 = " ".join(words[:mid])
    part2 = " ".join(words[mid:])
    return f"{part1}\n{part2}"

def plot_monopoly_heatmap(registros, use_simplified_board=False, save_path="monopoly_heatmap.png"):
    """Create a static Monopoly board heatmap with a polished aesthetic and larger dimensions."""
    # Use standard Monopoly board
    board = get_monopoly_board()
    board_size = len(board)  # 40
    landing_counts = compute_landing_frequencies(registros, board_size)
    
    # Identify high-frequency spaces (top 25% for heatmap)
    landing_counts_sorted = sorted(landing_counts, reverse=True)
    threshold_idx = int(len(landing_counts) * 0.25)  # Top 25%
    threshold = landing_counts_sorted[threshold_idx] if landing_counts_sorted else 0
    
    # Normalize counts for heatmap (only for spaces above threshold)
    max_count = max(landing_counts) if max(landing_counts) > 0 else 1
    normalized_counts = []
    for count in landing_counts:
        if count >= threshold:
            normalized = (count - threshold) / (max_count - threshold) if max_count > threshold else 0
        else:
            normalized = 0  # Spaces below threshold get no heatmap color
        normalized_counts.append(normalized)
    
    # Create board layout with larger uniform square spaces
    positions = []  # Center points for labels and path
    space_coords = []  # Coordinates for each space (x1, x2, y1, y2)
    idx = 0
    
    # Board dimensions: 11x11 grid (4 corners + 9 spaces per side = 11 spaces per side)
    space_size = 2.0  # Increased space size for larger board
    board_width = 11 * space_size  # Adjust board width proportionally
    
    # Bottom row (0-10): Go to Connecticut Avenue
    for x in range(11):
        x_pos = x * space_size
        positions.append((x_pos + space_size / 2, space_size / 2))
        space_coords.append((x_pos, x_pos + space_size, 0, space_size))
        idx += 1
    
    # Right column (11-20): Jail to New York Avenue
    for y in range(1, 11):
        y_pos = y * space_size
        positions.append((board_width - space_size / 2, y_pos + space_size / 2))
        space_coords.append((board_width - space_size, board_width, y_pos, y_pos + space_size))
        idx += 1
    
    # Top row (21-30): Free Parking to Marvin Gardens (reversed x)
    for x in range(9, -1, -1):
        x_pos = x * space_size
        positions.append((x_pos + space_size / 2, board_width - space_size / 2))
        space_coords.append((x_pos, x_pos + space_size, board_width - space_size, board_width))
        idx += 1
    
    # Left column (31-39): Go To Jail to Boardwalk (reversed y)
    for y in range(9, 0, -1):
        y_pos = y * space_size
        positions.append((space_size / 2, y_pos + space_size / 2))
        space_coords.append((0, space_size, y_pos, y_pos + space_size))
        idx += 1
    
    # Create the figure and axis
    fig, ax = plt.subplots(figsize=(12, 12))  # Larger figure size
    ax.set_xlim(-space_size / 2, board_width + space_size / 2 + 2)  # Extra space for colorbar
    ax.set_ylim(-space_size / 2, board_width + space_size / 2)
    ax.set_aspect('equal')
    ax.axis('off')
    
    # Background color
    fig.patch.set_facecolor("#D2B48C")  # Tan background
    
    # Define color mapping for properties
    color_map = {
        "brown": "#8B4513",
        "light_blue": "#ADD8E6",
        "pink": "#FF69B4",
        "orange": "#FFA500",
        "red": "#FF0000",
        "yellow": "#FFFF00",
        "green": "#008000",
        "dark_blue": "#00008B"
    }
    
    # Plot casillas with polished Monopoly styling
    for i, ((x, y), (x1, x2, y1, y2), space) in enumerate(zip(positions, space_coords, board)):
        space_type = space["type"]
        space_color = space["color"]
        label = space["name"]
        heatmap_intensity = normalized_counts[i]
        
        # Base color for the space
        base_color = "white"
        if space_type == "corner":
            base_color = "beige"
        
        # Apply heatmap overlay (red gradient for high frequency)
        if heatmap_intensity > 0:
            heatmap_opacity = heatmap_intensity * 0.9
            # Blend white/beige with red based on intensity
            base_rgb = np.array([1, 1, 1]) if base_color == "white" else np.array([245/255, 245/255, 220/255])
            red_rgb = np.array([1, 0, 0])
            blended_rgb = (1 - heatmap_opacity) * base_rgb + heatmap_opacity * red_rgb
            fill_color = blended_rgb
        else:
            fill_color = base_color
        
        # Draw the space
        if space_type == "property":
            # Determine orientation for color bar (0.2 * space_size units thick)
            color_bar_thickness = 0.2 * space_size
            if y1 == 0:  # Bottom row
                # Color bar on top
                rect_space = patches.Rectangle(
                    (x1, y1), space_size, space_size - color_bar_thickness,
                    linewidth=1, edgecolor='black', facecolor=fill_color
                )
                rect_color = patches.Rectangle(
                    (x1, y2 - color_bar_thickness), space_size, color_bar_thickness,
                    linewidth=1, edgecolor='black', facecolor=color_map.get(space_color, "white")
                )
            elif x1 == board_width - space_size:  # Right column
                # Color bar on left
                rect_space = patches.Rectangle(
                    (x1 + color_bar_thickness, y1), space_size - color_bar_thickness, space_size,
                    linewidth=1, edgecolor='black', facecolor=fill_color
                )
                rect_color = patches.Rectangle(
                    (x1, y1), color_bar_thickness, space_size,
                    linewidth=1, edgecolor='black', facecolor=color_map.get(space_color, "white")
                )
            elif y1 == board_width - space_size:  # Top row
                # Color bar on bottom
                rect_space = patches.Rectangle(
                    (x1, y1 + color_bar_thickness), space_size, space_size - color_bar_thickness,
                    linewidth=1, edgecolor='black', facecolor=fill_color
                )
                rect_color = patches.Rectangle(
                    (x1, y1), space_size, color_bar_thickness,
                    linewidth=1, edgecolor='black', facecolor=color_map.get(space_color, "white")
                )
            else:  # Left column
                # Color bar on right
                rect_space = patches.Rectangle(
                    (x1, y1), space_size - color_bar_thickness, space_size,
                    linewidth=1, edgecolor='black', facecolor=fill_color
                )
                rect_color = patches.Rectangle(
                    (x2 - color_bar_thickness, y1), color_bar_thickness, space_size,
                    linewidth=1, edgecolor='black', facecolor=color_map.get(space_color, "white")
                )
            
            ax.add_patch(rect_space)
            ax.add_patch(rect_color)
        else:
            # Non-property spaces (including corners)
            rect = patches.Rectangle(
                (x1, y1), space_size, space_size,
                linewidth=1, edgecolor='black', facecolor=fill_color
            )
            ax.add_patch(rect)
        
        # Add label with rotation
        display_label = split_label(label)
        text_angle = 0
        if x1 == board_width - space_size:  # Right column
            text_angle = -90
        elif y1 == board_width - space_size:  # Top row
            text_angle = 180
        elif x1 == 0:  # Left column
            text_angle = 90
        
        # Adjust position for properties with color bars
        label_x, label_y = x, y
        if space_type == "property":
            if y1 == 0:  # Bottom row
                label_y -= color_bar_thickness / 2  # Shift down to center in the space
            elif x1 == board_width - space_size:  # Right column
                label_x -= color_bar_thickness / 2  # Shift left
            elif y1 == board_width - space_size:  # Top row
                label_y += color_bar_thickness / 2  # Shift up
            elif x1 == 0:  # Left column
                label_x += color_bar_thickness / 2  # Shift right
        
        ax.text(
            label_x, label_y, display_label,
            ha='center', va='center',
            rotation=text_angle,
            fontsize=8, color='black',
            wrap=True
        )
    
    # Add a sample player's path (first player, first game, first 20 turns)
    if registros:
        sample_rolls = registros[0]['dice_rolls'][list(registros[0]['dice_rolls'].keys())[0]][:20]
        path_x = []
        path_y = []
        position = 0
        for turn_rolls in sample_rolls:
            if not turn_rolls:
                continue
            for roll in turn_rolls:
                move = sum(roll)
                position = (position + move) % board_size
                x, y = positions[position]
                path_x.append(x)
                path_y.append(y)
        
        ax.plot(path_x, path_y, color='blue', linestyle='--', linewidth=2, alpha=0.5, label='Sample Player Path')
        ax.scatter(path_x, path_y, color='blue', s=30, alpha=0.5)
    
    # Add central "MONOPOLY" label
    # Create a white background rectangle for the "MONOPOLY" text
    center_x, center_y = board_width / 2, board_width / 2
    text_bbox = dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5', alpha=0.8)
    ax.text(
        center_x, center_y, "MONOPOLY",
        ha='center', va='center',
        rotation=-45,
        fontsize=36, fontweight='bold', color='black',
        alpha=0.2,
        bbox=text_bbox
    )
    
    # Add colorbar for heatmap
    norm = plt.Normalize(0, 1)
    sm = plt.cm.ScalarMappable(cmap=plt.cm.Reds, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
    cbar.set_label('Frecuencia de Visitas', fontsize=12)
    cbar.set_ticks([0, 0.5, 1])
    cbar.set_ticklabels(['Baja', 'Media', 'Alta'])
    
    # Add title
    plt.title("Mapa de Calor de Monopoly: Frecuencia de Caídas por Casilla", fontsize=16, pad=20)
    
    # Save the figure as a high-resolution image
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Static heatmap saved as {save_path}")

In [341]:
# Genera el mapa de calor estático
plot_monopoly_heatmap(registros, use_simplified_board=False, save_path="monopoly_heatmap.png")


Static heatmap saved as monopoly_heatmap.png
